 # Prime testing

In [0]:
next_prime(2^200)


In [0]:
primes_first_n(10)


In [0]:
prime_range(100, 150)


In [0]:
next_prime(2000) * next_prime(1500)


In [0]:
prime_divisors(3026533)


 "Fast" prime testing using the Little Fermat theorem:

In [1]:
z = Integers(17449)
z(7814) ^ 17448


1

In [0]:
z = Integers(15485207)
z(2) ^ 15485206


 Now, let's use the form $a^p \equiv p \pmod{p}$:

In [0]:
n = 341


In [0]:
Integers(n)(2) ^ n


 **Question**: Does this prove that the above $n$ is a prime number?

 **Solution**: NO! The Little Fermat theorem only works in one direction: *if $p$ is a prime number, then $a^p \equiv a \pmod{p}$*

In [0]:
is_prime(n)


 If $n$ is a composite number, then $2^n \not\equiv 2 \pmod{n}$, otherwise *it is probably prime*. This is related to the following definition:



 **Definition**: Let $n \in \mathbb{Z}^+$ be a fixed integer. We say that the integer $a$ is a witness (that $n$ is composite) if $a^n \not\equiv a \pmod{n}$

In [0]:
def is_composite(n):
   z = Integers(n)
   return any(z(i)^n != z(i) for i in range(n))


In [0]:
[(i, is_composite(i)) for i in [137, 219, 561]]


 But there is a problem:

In [0]:
factor(561)


 The composite numbers for which $a^n \equiv a \pmod{n}$ holds are called Carmichael numbers. We need a stronger test.

 ## Miller-Rabin test



 **Theorem**: Let $p$ be an odd prime number and $p-1 = 2^kq$, where $q$ is an odd number. Let $a$ be an integer not divisible by $p$. Then one of the following two conditions holds:

 1. $a^q \equiv 1 \pmod{p}$

 2. one of $a^q,a^{2q},a^{4q},\dots,a^{2^{k-1}q}$ is congruent to $-1 \mod{p}$



 **Definition**: Let $n$ be an odd number and $n-1 = 2^kq$, where $q$ is an odd number. An integer $a$, for which $\text{gcd}(a,n)=1$, is called a Miller-Rabin witness for $n$ if the following holds:

 1. $a^q \not\equiv 1 \pmod{n}$

 2. $a^{2^iq} \not\equiv -1 \pmod{n}$, for all $i = 0, 1, 2, \dots, k-1$.



 That is, if there exists such an $a$, then $n$ is certainly a composite number.



 **Lemma**:

 If $n$ is a prime, then the only square roots of 1 modulo $n$ are 1 and $-1$.



 **Proof**:



 Certainly 1 and $-1$, when squared modulo $n$, always yield 1. It remains to show that there are no other square roots of 1 modulo $n$. This is a special case, here applied with the polynomial $X^2 - 1$ over the finite field $\mathbb{Z}/n\mathbb{Z}$, of the more general fact that a polynomial over some field has no more roots than its degree (this theorem follows from the existence of an Euclidean division for polynomials). Here follows a more elementary proof.



 Suppose that $x$ is a square root of 1 modulo $n$. Then:



 $$(x - 1)(x + 1) = x^2 - 1 \equiv 0 \pmod{n}.$$



 In other words, $n$ divides the product $(x - 1)(x + 1)$. By Euclid's lemma, since $n$ is prime, it divides one of the factors $x - 1$ or $x + 1$, implying that $x$ is congruent to either 1 or $-1$ modulo $n$.



 **Lemma**:

 If $n$ is an odd prime, then it is a strong probable prime to base $a$.



 **Proof**:



 If $n$ is an odd prime and we write $n - 1 = 2^sd$ where $s$ is a positive integer and $d$ is an odd positive integer, by Fermat's little theorem:



 $$a^{2^sd} \equiv 1 \pmod{n}.$$



 Each term of the sequence $a^{2^sd}, a^{2^{s-1}d}, \dots, a^{2d}, a^d$ is a square root of the previous term. Since the first term is congruent to 1, the second term is a square root of 1 modulo $n$. By the previous lemma, it is congruent to either 1 or $-1$ modulo $n$. If it is congruent to $-1$, we are done. Otherwise, it is congruent to 1 and we can iterate the reasoning. At the end, either one of the terms is congruent to $-1$, or all of them are congruent to 1, and in particular the last term, $a^d$, is.



 From this, the Miller-Rabin test for composite numbers can be built:



 **Input**: $n$ the integer we are testing and $a$ a potential witness

 1. If $n$ is even or $1 < \text{gcd}(a,n) < n$, then $n$ is **composite**.

 2. $n - 1 = 2^kq$, where $q$ is an odd number

 3. Let $a = a^q \pmod{n}$

 4. If $a \equiv 1 \pmod{n}$, return that **the test failed**

 5. For each $i = 0, 1, 2, \dots, k - 1$:

     - If $a \equiv -1 \pmod{n}$, return that **the test failed**

     - Let $a = a^2 \mod{n}$

 6. Return that **the number is composite**



 So, we generate many random $a$ numbers and check the result of the Miller-Rabin test. The following theorem will be helpful:



 **Theorem**: Let $n$ be an odd composite number. Then at least 75% of the numbers $1 \leq a \leq n-1$ are Miller-Rabin witnesses for $n$.

In [0]:
def miller_rabin(a: int, n: int) -> bool:
    if n % 2 == 0:
        return True
    k = 0
    q = n - 1
    while q % 2 == 0:
        q //= 2
        k += 1
    a = pow(a, q, n)
    if a % n == 1:
        return False
    for _ in range(k):
        if a % n == n - 1:
            return False
        a = pow(a, 2, n)
    return True


In [0]:
# print("\n".join(f"{n:>2}: {sum(miller_rabin(a, n) for a in range(2, n)) / n:.2f}" for n in range(2, 100)))
limit = 1000
from math import log
import matplotlib.pyplot as plt
plt.plot([n for n in range(2, limit) if any(miller_rabin(a, n) for a in range(2, n))], list(filter(bool, (sum(miller_rabin(a, n) for a in range(2, n)) / n for n in range(2, limit)))))
plt.plot([0, limit], [1, 1], linestyle="--")


 ## How many prime numbers are in the `[a..b]` interval?



 **Definition**: $\pi(x)$ gives the number of prime numbers in the `[2..x]` interval

In [0]:
len(prime_range(900_000, 1_000_000))


 And what about the interval $2^{1023} < p < 2^{1024}$? There are a lot:



 $$\pi(2^{1024}) - \pi(2^{1023}) \approx \frac{2^{1024}}{\ln{2^{1024}}} - \frac{2^{1023}}{\ln{2^{1023}}} \approx 2^{1013.53}$$

 **Task**: Find a 1024-bit long (about 300-digit) prime number! (In reality, even larger ones are needed)

In [0]:
next_prime(2^1023)


 # Factorization

 ## Pollard's $p-1$ algorithm



 Let $p,q$ be prime numbers and $N=pq$ (we only know $N$). **Task**: Determine $p$ and $q$!



 With some effort, we found an integer $L$ such that $p-1$ divides $L$ and $q-1$ does not divide $L$. Then there exist integers $i,j,k$ ($k \ne 0$) such that $L = i(p-1)$ and $L = j(q-1)+k$.



 Let's take an arbitrary integer $a$ and calculate the value of $a^L$:

 \begin{align*}

     a^L &= a^{i(p-1)} = (a^{p-1})^i \equiv 1^i \equiv 1 \pmod{p} \\

     a^L &= a^{j(q-1)+k} = a^k (a^{q-1})^j \equiv a^k \cdot 1^j \equiv a^k \pmod{q}

 \end{align*}



 Since $p$ and $q$ are very large and $k\ne 0$, with high probability $p$ will divide $a^L -1$ and $q$ will not divide $a^L -1$. From this, we can easily get $p$: $p = \text{gcd}(a^L - 1, N)$ and we're done.

 Where do we find such an $L$? Pollard: if $p-1$ can be expressed as a product of small prime numbers, then it will divide $n!$ for not too large $n$. The method:

 - take $n=2,3,\dots$ numbers and calculate $\text{gcd}(a^{n!}-1,N)$

     - in practice, $a=2$

 - if the gcd is $1$, increase $n$ by one

 - if the gcd is $N$, choose another $a$

 - if $1 < \text{gcd} < N$, we have a non-trivial factorization of $N$ and we're done

 **Lesson**: The factorization of $p-1$ and $q-1$ is related to the factorization of $N$! Choose $p$ and $q$ such that $p-1$ will not be expressible as a product of small prime numbers.

In [0]:
N = 64384091
a = 2
n = 1
while True:
    L = factorial(n)
    d = gcd(a^L - 1, N)
    if 1 < d < N:
        print(f"Found a factor: {d} (n = {n})")
        break
    n += 1


 ### Pollard's $\rho$ algorithm



 Let $S$ be a finite set and $f: S \rightarrow S$ a function. Suppose we apply the $f$ function repeatedly to an element $x \in S$, which results in the following sequence:

 $$

 x_0 = x,\quad x_1 = f(x_0),\quad x_2 = f(x_1),\quad x_3 = f(x_2),\quad \dots,

 $$

 that is, $x_i = (\underbrace{f \circ f \circ f \circ \cdots \circ f}_\text{i-times})(x)$



 Since $S$ is a finite set, there will eventually be a repetition. For a while, $x_0,x_1,x_2,\dots$ will move in one direction, with no repeats (this length is $T$). After a while ($M$ steps), the elements will start repeating.



 ![Pollard's rho algorithm](./pollard_rho.png)



 Let the size of $S$ be $N$. Then the length of $T + M$ is roughly $1.2533 \sqrt{N}$. Since we don't know the values of $T$ and $M$ in advance, we need to generate the $x_0, x_1, x_2, \dots, x_{M + T}$ sequence.



 To find matching elements



 Pollard: compute two sequences

 - the above $x_i$ sequence, and

 - a $y_i$ sequence such that $y_0 = x_0$ and $y_{i+1} = f\left( f(y_i) \right)$ for all $i = 0,1,2,\dots$, that is, $y_i = x_{2i}$



 When do we find a match (*collision*)? In general, for $j > i$, $x_j = x_i$ if and only if $i \geq T$ and $j \equiv i \pmod{M}$. Specifically: $x_{2i} = x_i$ if and only if $i \geq T$ and $2i \equiv i \pmod{M}$. The latter implies $M\;|\;i$, that is, they match when $i$ is equal to the first multiple of $M$ greater than $T$.

 #### Example of Pollard's $\rho$ algorithm



 **Prime factorization**: Let $p,q$ be $n$-bit prime numbers, $N = pq$ and $f : \mathbb{Z}_N^* \rightarrow \mathbb{Z}_N^*$. Choose an arbitrary $x_0 \in \mathbb{Z}_N^*$ and let $f(x) = (x^2 + 1 \mod{N})$. Then the following algorithm will produce one of the prime factors of $N$:



 1. $x_0 \leftarrow \mathbb{Z}_N^*$

 2. $x' := x := x_0$

 3. **for** $i = 1$ **to** $2^{n/2}$:

     - $x := f(x)$

     - $x' := f(f(x'))$

     - $p := \text{gcd}(x - x', N)$

     - **if** $p \not\in \{1, N\}$ **return** p and stop

 **Example**: Find the prime factorization of $N = 37241$!

In [0]:
N = 37241
f = lambda x: (x^2 + 1) % N
x = 2
y = x
d = 1
for i in range(1, 2^floor(log(N, 2) / 2)):
    x = f(x)
    y = f(f(y))
    d = gcd(x - y, N)
    if d != 1:
        break
d


 **Example**: Use Pollard's $\rho$ algorithm to solve the discrete logarithm problem $19^t \equiv 24717 \pmod{48611}$!

In [0]:
p = 48611
z = Zmod(p)
g = z(19)
h = 24717


In [0]:
def f(x, i, j):
   x = int(x)
   if 0 <= x < p/3:
       return int((g * x) % p), int((i+1) % p), int(j % p)
   elif p/3 <= x < 2*p/3:
       return int(pow(x, 2, p)), int((2*i) % p), int((2*j) % p)
   elif 2*p/3 <= x < p:
       return int((h * x) % p), int(i % p), int((j+1) % p)


In [0]:
x, y = 1, 1
a, b, c, d = 0, 0, 0, 0
for i in range(550):
   print(f'{i:>3d} {x:>5d} {y:>5d} {a:>5d} {b:>5d} {c:>5d} {d:>5d}')
   x, a, b = f(x, a, b)
   y, c, d = f(*f(y, c, d))


In [0]:
mod(-35140, 48610)


In [0]:
-35140 + 48610


In [0]:
mod(19 ^ -35140, p), mod(19 ^ 13470, p)